In [1]:
import pandas as pd
import numpy as np


In [ ]:
data = pd.read_csv('dst-3.0_16_1_hh_database.csv', sep = ';')

In [ ]:
def get_education(arg):
    arg = ' '.join(arg.split(' ')[:3])
    if 'Высшее' in arg:
        return 'высшее'
    elif 'Неоконченное высшее' in arg:
        return 'неоконченное высшее'
    elif 'Среднее специальное' in arg:
        return 'среднее специальное'
    elif 'Среднее образование' in arg:
        return 'среднее'
data['Образование'] = data['Образование и ВУЗ'].apply(get_education)
#data = data.drop('Образование и ВУЗ', axis=1)
print(data['Образование'].value_counts()['среднее'])



In [ ]:
education = {
    'Высшее': 'высшее',
    'Неоконченное': 'неоконченное высшее',
    'Среднее специальное': 'среднее специальное',
    'Среднее образование': 'среднее'
}

def type_education(df, sourse_col='Образование и ВУЗ', target_col='Образование'):
    
    text = df[sourse_col].fillna('').astype(str)
    
    result = pd.Series('не указано', index=df.index, dtype='object')
    
    for initial_value, final_value in education.items():
        mask = text.str.startswith(initial_value)
        result[mask] = final_value
        
    result[df[sourse_col].isna()] = 'Не указано'
    
    return result

data['Образование'] = type_education(data)
#data = data.drop('Образование и ВУЗ', axis=1)

data['Образование'].value_counts()

In [ ]:
def identification(x):
    x = x.split(',')[0]
    if 'Мужчина' in x:
        return 'М'
    elif 'Женщина' in x:
        return 'Ж'

def search_age(x):
    words = ["год", "года", "лет"]
    parts = x.split()
    for i, word in enumerate(parts):
        if word in words and i > 0:
            return int(parts[i-1])

data['Sex'] = data['Пол, возраст'].apply(identification)
data['Age'] = data['Пол, возраст'].apply(search_age)

data['Sex'].value_counts(normalize=True) * 100
data['Age'].mean()

In [ ]:
data['Sex'] = data['Пол, возраст'].str.extract(r'(Мужчина|Женщина)')[0].map({
    'Мужчина': 'М', 'Женщина': 'Ж'
})

data['Age'] = pd.to_numeric(
    data['Пол, возраст'].str.extract(r'(\d+)')[0],
    errors='coerce', downcast='integer'
)

In [ ]:
def type_education(df, source_col='Образование и ВУЗ', target_col='Образование'):
    
    text = (
        df[source_col]
        .fillna('')
        .astype(str)
        .str.split(' ')
        .str[:2]
        .str.join(' ')
    )
    
    replacements = {
        r'^Высшее образование.*': 'высшее',
        r'^Неоконченное высшее.*': 'неоконченное высшее',
        r'^Среднее специальное.*': 'среднее специальное',
        r'^Среднее образование.*': 'среднее'
    }
    
    result = text.replace(replacements, regex=True)
    result[df[source_col].isna()] = 'Не указано'
    return result


data['Образование'] = type_education(data)

data.drop('Образование и ВУЗ', axis = 'columns') #Временное удаление колонки, чтобы не мешалась при извлечении пола и возраста

data['Образование'].head()

In [ ]:
import pandas as pd
import numpy as np
import re

data = pd.read_csv('dst-3.0_16_1_hh_database.csv', sep = ';')

In [ ]:
years = data['Опыт работы'].str.extract(r'(\d+)\s*(?:год|года|лет)', flags=re.I)[0]

months = data['Опыт работы'].str.extract(r'(\d+)\s*(?:месяц|месяца|месяцев|мес)', flags=re.I)[0]

# Переводим в числа (ошибки станут NaN)
years = pd.to_numeric(years, errors='coerce')
months = pd.to_numeric(months, errors='coerce')

# Считаем месяцы (NaN заменяем на 0)
data['Опыт работы (месяцев)'] = years.fillna(0) * 12 + months.fillna(0)

# Если не было ни лет, ни месяцев - ставим NaN
data.loc[(years.isna() & months.isna()), 'Опыт работы (месяцев)'] = np.nan

# Если было 'Не указано' - ставим NaN
data.loc[data['Опыт работы'].astype(str).str.contains('Не указано', case=False, na=False), 'Опыт работы (месяцев)'] = np.nan

# Удаляем исходную колонку
# data = data.drop('Опыт работы', axis=1)

# Результат
print(f"Медианный опыт: {data['Опыт работы (месяцев)'].median():.1f} месяцев")
display(data['Опыт работы (месяцев)'].median())


In [ ]:
import pandas as pd
import numpy as np
import re

def extract_experience_months(df, source_col='Опыт работы', target_col='Опыт работы (месяцев)'):
    """
    Извлекает опыт работы в месяцах из текстового поля
    Оптимизированная версия
    """
    # 1. Эффективное извлечение лет и месяцев (один проход)
    experience_text = df[source_col].fillna('').astype(str)
    
    # Извлекаем годы (сразу как числа)
    years = experience_text.str.extract(r'(\d+)\s*(?:лет|год|года)', flags=re.I, expand=False)
    years_num = pd.to_numeric(years, errors='coerce').fillna(0)
    
    # Извлекаем месяцы
    months = experience_text.str.extract(r'(\d+)\s*(?:месяц|месяца|месяцев)', flags=re.I, expand=False)
    months_num = pd.to_numeric(months, errors='coerce').fillna(0)
    
    # 2. Вычисляем общее количество месяцев
    total_months = years_num * 12 + months_num
    
    # 3. Обрабатываем особые случаи
    is_missing = df[source_col].isna()
    is_empty = experience_text.str.strip().isin(['', 'Не указано', 'nan', 'None'])
    
    # 4. Устанавливаем NaN для пропусков
    total_months[is_missing | is_empty] = np.nan
    
    # 5. Сохраняем результат
    df[target_col] = total_months
    
    # 6. Удаляем исходную колонку
    df.drop(source_col, axis='columns', inplace=True)
    
    return df

# Использование
data = extract_experience_months(data)
data['Опыт работы (месяцев)'].median()


In [ ]:
import pandas as pd
import numpy as np
import re
data = pd.read_csv('dst-3.0_16_1_hh_database.csv', sep = ';')

In [ ]:
text = data['Город, переезд, командировки'].fillna('').str.lower()

# --- ГОРОД ---
city = (
    text
    .str.replace(r'\s*\([^)]*\)', '', regex=True)  # убрать скобки
    .str.split(',')
    .str[0]
    .str.strip()
)

million_cities = ['новосибирск', 'екатеринбург', 'нижний новгород',
                  'казань', 'челябинск', 'омск', 'самара',
                  'ростов-на-дону', 'уфа', 'красноярск',
                  'пермь', 'воронеж', 'волгоград']

data['Город'] = 'другие'

data.loc[city.isin(['москва']), 'Город'] = 'Москва'
data.loc[city.isin(['санкт-петербург', 'спб', 'питер']), 'Город'] = 'Санкт-Петербург'
data.loc[city.isin(million_cities), 'Город'] = 'город миллионник'

data.loc[text.eq(''), 'Город'] = np.nan


# Переезд
data['Готовность к переезду'] = (
    text.str.contains(r'(?:готов|хочу)', na=False) &
    ~text.str.contains(r'(?:не)\s+.*(?:готов)', na=False)
)

# Командировки
data['Готовность к командировкам'] = (
    text.str.contains(r'готов', na=False) &
    ~text.str.contains(r'(?:не)\s+.*(?:готов)', na=False)
)

# --- УДАЛЕНИЕ СТОЛБЦА ---
#data = data.drop('Город, переезд, командировки', axis=1)

print(round(data['Город'].value_counts(normalize=True)['Санкт-Петербург'] * 100)) 
print(round(data[
    data['Готовность к переезду'] & data['Готовность к командировкам']
].shape[0] / data.shape[0] *100))

In [ ]:
# Все возможные категории занятости и графика
employment_cats = ['полная занятость', 'частичная занятость', 'проектная работа', 
                    'волонтерство', 'стажировка']

schedule_cats = ['полный день', 'сменный график', 'гибкий график', 
                  'удаленная работа', 'вахтовый метод']

def create_features(df, col_name, categories):
    text = df[col_name].fillna('').astype(str)
    
    for cat in categories:
        df[f'{col_name}_{cat}'] = text.str.contains(cat, na=False, case=False, regex=False)
        
    return df

data = create_features(data, 'Занятость', employment_cats)
data = create_features(data, 'График', schedule_cats)

display(data.filter(like='Занятость').head())
display(data.filter(like='График').head())

display(data.iloc[:, 12::].head())


In [ ]:
# Загрузка курсов
rates_df = pd.read_csv('ExchangeRates.csv')
currency_dict = {'руб.':'RUB', 'KZT':'KZT', 'USD':'USD', 'бел.руб.':'BYN', 'EUR':'EUR', 'грн.':'UAH', 'сум':'UZS', 'KGS':'KGS', 'AZN':'AZN'} 

# Приведение дат к единому формату
data['Обновление резюме'] = pd.to_datetime(data['Обновление резюме'], dayfirst=True).dt.date
rates_df['date'] = pd.to_datetime(rates_df['date'], format='%d/%m/%y').dt.date

# Извлечение зарплаты и валюты
data['Зп'] = data['ЗП'].str.extract(r'(\d[\d\s]*)')[0].str.replace(r'\s', '', regex=True).astype(float)

data['Валюта'] = data['ЗП'].str.extract(r'(\D+)$')[0].str.strip().map(currency_dict).fillna('RUB')

# Объединяем данные с курсами валют
data = data.merge(
    rates_df,
    left_on=['Обновление резюме', 'Валюта'],
    right_on=['date', 'currency'],
    how='left'
)

# Для рубля устанавливаем курс 1 и пропорцию 1, так как это базовая валюта
data['close'] = data['close'].fillna(1)
data['proportion'] = data['proportion'].fillna(1)

# Считаем зарплату в рублях
data['ЗП_руб'] = data['Зп'] * data['close'] / data['proportion']

#Временное удаление колонок, которые больше не нужны
data = data.drop(['ЗП', 'Валюта', 'date', 'currency', 'close', 'proportion'], axis=1)

# Проверяем результат
data[['ЗП_руб']].head()

In [ ]:
text = data['Город, переезд, командировки'].fillna('').str.lower()

# Извлекаем город
city = (
    text
    .str.replace(r'\s*\([^)]*\)', '', regex=True)
    .str.split(',')
    .str[0]
    .str.strip()
)

million_cities = ['новосибирск', 'екатеринбург', 'нижний новгород',
                  'казань', 'челябинск', 'омск', 'самара',
                  'ростов-на-дону', 'уфа', 'красноярск',
                  'пермь', 'воронеж', 'волгоград']

data['Город'] = 'другие'

data.loc[city.isin(['москва']), 'Город'] = 'Москва'
data.loc[city.isin(['санкт-петербург', 'спб', 'питер']), 'Город'] = 'Санкт-Петербург'
data.loc[city.isin(million_cities), 'Город'] = 'город миллионник'

data.loc[text.eq(''), 'Город'] = np.nan


# Переезд
data['Готовность к переезду'] = (
    text.str.contains(r'\b(?:готов|готова|хочу)', na=False) &
    ~text.str.contains(r'(?:не)\s+.*(?:готов, готова)', na=False)
)

# Командировки
data['Готовность к командировкам'] = (
    text.str.contains(r'готов', na=False) &
    ~text.str.contains(r'(?:не)\s+.*(?:готов)', na=False)
)

# Временное удаление колонки
data = data.drop('Город, переезд, командировки', axis=1)

print(round(data['Город'].value_counts(normalize=True)['Санкт-Петербург'] * 100)) 
print(round(data[
    data['Готовность к переезду'] & data['Готовность к командировкам']
].shape[0] / data.shape[0] *100))


In [ ]:
import numpy as np

def any_normal(*vectors):
    
    vectors = [np.array(v) for v in vectors]
    n = len(vectors)
    
    for i in range(n):
        
        for j in range(i+1, n):
              
              dot_product = np.dot(vectors[i], vectors[j])


    return False


def any_normal(*vectors):
    for i in range(len(vectors)):
        for j in range(i+1, len(vectors)):
            if np.dot(vectors[i], vectors[j]) == 0:
                return True
    return False
# i=0: j=0 → (v1, v1) — ❌ вектор с самим собой
# i=0: j=1 → (v1, v2) — ✅ первая пара
# i=0: j=2 → (v1, v3) — ✅ вторая пара

def get_loto(num):

    loto_arrays = np.random.randint(1, 101, size=(num, 5, 5))

    return loto_arrays


def get_unique_loto(num):
    
    result = np.zeros((num, 5, 5), dtype=int)
    
    for i in range(num):
        
        unique_numbers = np.random.choice(
            range(1, 101),
            size=25,
            replace=False
        )
        
        result[i] = unique_numbers.reshape(5, 5)
        
    return result

SyntaxError: Missing parentheses in call to 'print'. Did you mean print(...)? (2095745209.py, line 3)